In [1]:
import glob
from scalesurfer.config import DATA_PATH, MODULE_PATH, SEED
from scalesurfer.experiments import build_v8_split_from_root
from scalesurfer import ScaleSurfer

In [2]:
# load dataset and model
BASE_PATH = MODULE_PATH.parent
FS_VERSION_JSON = BASE_PATH / "data" /"fs_version_by_dataset_updated.json"
TENSORS_ROOT_V8 = BASE_PATH / "data" / "tensors_gcloud"
RANDOM_SEED = int(SEED)
SPLIT_RATIOS = (0.8, 0.1, 0.1)
split_v8 = build_v8_split_from_root(
    tensors_root=TENSORS_ROOT_V8,
    seed=RANDOM_SEED,
    ratios=SPLIT_RATIOS
)

# test set
ds = [i[42:50] for i in split_v8["x_test"]]
nii = glob.glob("/home/rph/scalesurfer/data/openneuro_test_set_raw/files/**/*.ni*", recursive=True)
nii = {i[56:64]: i for i in nii}
ds = {i: nii[i] for i in ds}

In [ ]:
images = list(ds.values())[:3]
images

['/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds002168/sub-1516/anat/sub-1516_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds003720/sub-005/anat/sub-005_T1w.nii',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds001241/sub-14/anat/sub-14_T1w.nii.gz']

In [4]:
subjects = list(ds.keys())[:3]
subjects

['ds002168', 'ds003720', 'ds001241']

In [5]:
# Initialize object
surfer = ScaleSurfer(
    images,
    subjects,
    DATA_PATH / "subjects_dir",
    in_memory=True, # store all tensors in memory
    device="cuda"
)

In [6]:
surfer.prepare_images() # saves subject_dir/*/mri/rawavg.pt

Converting niftis to tensors:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
surfer.predict_volumes() # saves subject_dir/*/mri/aparc+aseg.pt

Predicting volumes:   0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
surfer.predict_surfaces() # save subject_dir/*/surf/{lh.white, rh.white, lh.pial, rh.pial}

Predicting surfaces:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# todo: implement a plotting method
# surfer.plot_volume(idx) # plots image + aparc+aseg
# todo: run on full test dataset and make plot for paper